# Module 1.2: Probability & Calculus

In the previous notebook, we looked at how to represent words as math (Vectors & dot products). But how does the Transformer know *which* numbers to put in those vectors so that it gives good answers? This notebook explores the engine of learning: **Loss, Gradients, and Optimization**.

## 1. Logits & Probabilities: The Model's Guesses

### The Concept
Before a Transformer speaks, the final linear layer spits out raw scores for every word in its vocabulary. These raw, un-normalized numbers are called **Logits**.

### The Math
We pass these Logits through the **Softmax** function to convert them into **Probabilities** (0.0 to 1.0) so we can see which word is the 'favorite'.
$$\sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum e^{z_j}}$$

### Why?
The model needs to express its confidence in a standard way (summing to 100%). We cannot easily optimize raw, unbounded logits directly because we wouldn't know if a score of '5.0' means the model is completely certain or just slightly more certain than '4.9'.

In [ ]:
import torch
import torch.nn.functional as F

# Reproducibility: fix the random seed so every run gives the same numbers.
torch.manual_seed(0)

# Imagine our vocabulary only has 3 words: [Apple, Banana, Cat]
logits = torch.tensor([2.5, -1.0, 5.0]) # Raw scores

probs = F.softmax(logits, dim=0)
print(f"Probabilities: {probs}")
print(f"The model is {probs[2]*100:.1f}% sure the word is 'Cat'")

## 2. Cross-Entropy Loss (The "Surprise" Metric)

### The "Weather Forecaster" Analogy
Imagine a weather forecaster says there is a 99% chance of rain. If it is sunny, the forecaster's **surprise** (and the penalty from their boss) is massive. If they said 50%, their surprise is moderate. 

**Loss** is how we measure this surprise. We want the loss to be as low as possible.

### The Math
$$L = -\sum_i y_i \log(p_i)$$
- $y_i$: The true answer (1 for the correct word, 0 for others).
- $p_i$: The model's predicted probability for that word.

**A key simplification:** because $y$ is **one-hot** (exactly one entry is 1, all the rest are 0), every term in the sum is multiplied by 0 *except* the single correct word. So the whole formula collapses to:
$$L = -\log(p_{\text{correct}})$$
In words: cross-entropy loss is just the **negative log-probability the model assigned to the right answer**. Assign high probability to the correct word → small loss; assign low probability → big loss.

We use $\log$ because: 
1. It heavily penalizes being *confident but wrong*.
2. Adding logs is numerically more stable than multiplying tiny probabilities.

In [ ]:
target_index = torch.tensor(2) # The right answer was 2 ("Cat")

# Note: PyTorch cross_entropy expects raw logits, not probabilities! It applies log_softmax inside.
loss = F.cross_entropy(logits.unsqueeze(0), target_index.unsqueeze(0))
print(f"Surprise (Loss) for guessing right: {loss.item():.4f}")

# Confirm the one-hot simplification L = -log(p_correct):
manual = -torch.log(probs[2])
print(f"Manual -log(p_correct):             {manual.item():.4f}  (matches above)")

# What if the answer was actually "Banana" (Index 1)?
bad_target = torch.tensor(1)
bad_loss = F.cross_entropy(logits.unsqueeze(0), bad_target.unsqueeze(0))
print(f"Surprise (Loss) for missing completely: {bad_loss.item():.4f}")

## 3. Calculus: Gradients (The Compass)

### The "Blindfolded Mountaineer" Analogy
If you are stuck on a mountain with a blindfold and want to get to the very bottom (lowest Loss), you can't see the bottom. But you can feel the slope of the ground beneath your feet. A **Gradient** is that slope.

### The Math
A derivative $\frac{\partial L}{\partial w}$ tells us: *"If I tweak this weight $w$ slightly, how much will my Loss $L$ change?"*
If the gradient is positive, increasing the weight makes the loss go up (bad). So we must go the opposite way.

### Why?
Without gradients, the model would have to guess weights randomly across billions of parameters. The gradient gives us the exact mathematical direction to move to improve our answers.

In [ ]:
# Let's make a single weight.
# requires_grad=True tells PyTorch: "Track everything that happens to this number!"
w = torch.tensor(3.0, requires_grad=True)

# Imagine our loss function is a simple curve: L = w^2
loss = w**2

# Calculate the slope (derivative)
loss.backward()

print(f"Weight value: {w.item()}")
print(f"Loss value: {loss.item()}")
print(f"Gradient (Slope): {w.grad.item()}") # Since L = w^2, dL/dw = 2w = 6.0

## 4. The Chain Rule & Backpropagation (The Blame Game)

### The "Factory" Analogy
Imagine a factory line: Robot A mixes the paint, Robot B sprays it. The final car looks terrible. Who is to blame? 
**Backpropagation** is the process of examining the final error, blaming the sprayer (B) slightly, and then the sprayer blaming the mixer (A). We propagate the error backward.

### The Math (Chain Rule)
$$\frac{\partial z}{\partial x} = \frac{\partial z}{\partial y} \cdot \frac{\partial y}{\partial x}$$

Because LLMs are just giant stacks of matrices, the chain rule allows the Loss at the very end to pass gradients all the way back to the very first Embedding layer.

### Why?
An LLM has dozens of layers. If the final output is wrong, we need a mathematical way to figure out exactly how much the *very first* layer contributed to that mistake, so we can correct it. Backpropagation solves this blame-assignment problem.

In [ ]:
# Layer 1 (Mixer)
robot_a = torch.tensor(2.0, requires_grad=True)
# Layer 2 (Sprayer): b = a * 3
robot_b = robot_a * 3.0
# Final Loss: error = b ** 2
error = robot_b ** 2

# The blame game
error.backward()

# Let's verify PyTorch's number by doing the chain rule BY HAND.
# We want d(error)/d(a). The chain links two steps:
#   error = b^2     ->  d(error)/db = 2*b   = 2 * 6.0 = 12.0
#   b     = a * 3   ->  d(b)/da     = 3
# Chain rule multiplies the links:
#   d(error)/da = d(error)/db * d(b)/da = 12.0 * 3 = 36.0
b_val = (robot_a * 3.0).item()
d_error_db = 2 * b_val      # = 12.0
d_b_da = 3                  # = 3
chain = d_error_db * d_b_da # = 36.0

print(f"Chain rule by hand: d(error)/db * d(b)/da = {d_error_db} * {d_b_da} = {chain}")
print(f"Robot A gets this much blame (PyTorch):    {robot_a.grad.item()}")

## 5. Gradient Descent (Taking the Step)

### The Concept
Once we know the slope (gradient), we step down the mountain. 
$$ W_{new} = W_{old} - \eta \cdot \text{gradient} $$

- $\eta$ (eta) is the **Learning Rate**.
- If $\eta$ is too small: We take baby steps and take millions of years to train.
- If $\eta$ is too big: We leap across the valley and miss the bottom entirely.

> **Note on LLMs**: Standard Gradient Descent (SGD) is simple, but LLMs use **AdamW**, which adds three things on top of plain SGD:
> 1. **Momentum (1st moment):** it keeps a running average of past gradients, so if we're rolling steadily down a hill we keep some of that speed.
> 2. **Per-parameter adaptive step sizes (2nd moment):** it also tracks how *big and noisy* each parameter's gradients have been, and shrinks the step for jittery parameters while letting calm ones move faster — so every one of the billions of weights effectively gets its own tuned learning rate.
> 3. **Decoupled weight decay (the "W"):** it gently pulls weights toward zero *separately* from the gradient step, which regularizes the model and is what distinguishes AdamW from plain Adam.

### Why?
This is the actual mechanism of 'learning'. By repeatedly finding the slope and taking a tiny step, the model slowly sculpts its matrices to capture patterns, grammar, and facts hidden in the training data.

In [ ]:
# Let's do 5 steps of learning to reach the bottom (loss = 0)
w = torch.tensor(10.0, requires_grad=True)
learning_rate = 0.1

for step in range(5):
    loss = w**2        # Forward pass
    loss.backward()    # Backprop (find gradient)
    
    # We use torch.no_grad so our weight update isn't tracked as part of the math
    with torch.no_grad():
        w -= learning_rate * w.grad
        w.grad.zero_() # Important: PyTorch adds gradients together by default. Reset to 0!
        
    print(f"Step {step+1}: w = {w.item():.2f}, loss = {loss.item():.2f}")

### 🏋️ Try it yourself

1. **Feel the loss.** Using `F.cross_entropy` with the logits `[2.5, -1.0, 5.0]`, compute the loss for *each* of the three possible correct answers (index 0, 1, and 2). Which target gives the smallest loss, and why? Confirm one of them by hand with `-torch.log(p_correct)`.
2. **Train a single weight.** Start from `w = 5.0` and run gradient descent on `loss = (w - 2) ** 2` for 10 steps with a learning rate of `0.1`. Print `w` each step and check that it converges toward `2.0` (the minimum). Try a learning rate of `1.1` and observe what goes wrong.

In [ ]:
import torch
import torch.nn.functional as F

# --- Exercise 1: loss for each possible target ---
logits = torch.tensor([2.5, -1.0, 5.0])
# TODO: loop over targets 0, 1, 2 and print F.cross_entropy for each.
# Which is smallest? (Hint: the target whose logit is highest.)


# --- Exercise 2: train a single weight toward 2.0 ---
w = torch.tensor(5.0, requires_grad=True)
learning_rate = 0.1
for step in range(10):
    loss = (w - 2) ** 2
    loss.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad
        w.grad.zero_()
    # TODO: print step, w.item(), loss.item()
# Then try learning_rate = 1.1 and watch w diverge.

---

### Up Next

You now have both halves of the toolkit: **Module 1.1** showed how to *represent and compare* words with linear algebra and attention, and this notebook showed how the model *learns* — turning predictions into a loss, backpropagating gradients, and stepping downhill with AdamW. Next we put these pieces together to build the **tokenizer and embedding layer**, the first real component of our Transformer, and start feeding actual text through the machinery.